# Model 6: BiGRU with Pretrained Word2Vec Embeddings (Gensim)

Train Word2Vec on the IMDB corpus, then use those vectors to initialise the embedding layer of a BiGRU classifier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from gensim.models import Word2Vec
import re, collections
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)

In [ ]:
# ── Load & Clean ───────────────────────────────────────────────────────────
df = pd.read_csv('IMDB Dataset.csv')
df['label'] = (df['sentiment'] == 'positive').astype(int)

def clean(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return text.lower().split()

df['tokens'] = df['review'].apply(clean)
print('Sample:', df['tokens'][0][:10])

In [ ]:
# ── Train Word2Vec on the full corpus ─────────────────────────────────────
EMBED_DIM = 128
print('Training Word2Vec...')
w2v = Word2Vec(
    sentences=df['tokens'].tolist(),
    vector_size=EMBED_DIM,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram
    epochs=10,
)
print(f'W2V vocab size: {len(w2v.wv)}')
# Quick sanity check
print('Most similar to "great":', w2v.wv.most_similar('great', topn=5))

In [ ]:
# ── Build Vocabulary & Embedding Matrix ────────────────────────────────────
MAX_LEN = 256
PAD_IDX, UNK_IDX = 0, 1
vocab = ['<PAD>', '<UNK>'] + list(w2v.wv.key_to_index.keys())
w2i   = {w: i for i, w in enumerate(vocab)}

# Initialise embedding matrix: rows = vocab tokens, cols = embed_dim
embed_matrix = np.zeros((len(vocab), EMBED_DIM), dtype=np.float32)
for word, idx in w2i.items():
    if word in w2v.wv:
        embed_matrix[idx] = w2v.wv[word]

embed_tensor = torch.FloatTensor(embed_matrix)
print(f'Embedding matrix: {embed_tensor.shape}')

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
df['ids'] = df['tokens'].apply(
    lambda toks: [w2i.get(t, UNK_IDX) for t in toks[:MAX_LEN]]
)

class IMDBDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids    = [torch.LongTensor(x) for x in ids]
        self.labels = torch.LongTensor(labels)
    def __len__(self):  return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.labels[i]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    return pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX), torch.stack(labels)

train_idx, test_idx = train_test_split(range(len(df)), test_size=0.2, random_state=42, stratify=df['label'])
train_idx, val_idx  = train_test_split(train_idx, test_size=0.1, random_state=42)

def make_loader(idx, shuffle=True):
    ds = IMDBDataset(df['ids'].iloc[idx].tolist(), df['label'].iloc[idx].tolist())
    return DataLoader(ds, batch_size=64, shuffle=shuffle, collate_fn=collate_fn)

train_loader = make_loader(train_idx)
val_loader   = make_loader(val_idx,  shuffle=False)
test_loader  = make_loader(test_idx, shuffle=False)

In [ ]:
# ── BiGRU with Pretrained Embeddings ──────────────────────────────────────
class BiGRU_W2V(nn.Module):
    def __init__(self, embed_matrix, hidden_dim=128, n_layers=2,
                 dropout=0.3, freeze_emb=False):
        super().__init__()
        vocab_size, embed_dim = embed_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            embed_matrix, padding_idx=PAD_IDX, freeze=freeze_emb
        )
        self.gru = nn.GRU(
            embed_dim, hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x):
        emb    = self.dropout(self.embedding(x))
        _, h   = self.gru(emb)
        h      = torch.cat([h[-2], h[-1]], dim=1)   # last fwd + bwd
        return self.fc(self.dropout(h))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = BiGRU_W2V(embed_tensor, freeze_emb=False).to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training ───────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            n          += len(yb)
    return total_loss / n, correct / n

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val, best_state, patience_cnt, PATIENCE = 0, None, 0, 5

for epoch in range(1, 21):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    if va > best_val:
        best_val = va; patience_cnt = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}'); break
    print(f'Epoch {epoch:2d} | Train {ta:.4f} | Val {va:.4f}')

model.load_state_dict(best_state)

In [ ]:
# ── Evaluation & Save ─────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(yb.numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f'Test Accuracy: {test_acc:.4f}')
print(classification_report(all_labels, all_preds, target_names=['Negative','Positive']))

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('BiGRU+W2V Loss'); axes[0].legend()
axes[1].plot(history['train_acc'],  label='Train'); axes[1].plot(history['val_acc'],  label='Val')
axes[1].set_title('BiGRU+W2V Accuracy'); axes[1].legend()
plt.tight_layout(); plt.savefig('../results/w2v_curves.png', dpi=150); plt.show()

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Negative','Positive'], yticklabels=['Negative','Positive'])
plt.title('Confusion Matrix – BiGRU + Word2Vec')
plt.tight_layout(); plt.savefig('../results/w2v_confusion_matrix.png', dpi=150); plt.show()

report = classification_report(all_labels, all_preds, target_names=['Negative','Positive'])
with open('../results/w2v_results.txt', 'w') as f:
    f.write('=== BiGRU + Pretrained Word2Vec Results ===\n\n')
    f.write(f'Test Accuracy: {test_acc:.4f}\n\n')
    f.write('Classification Report:\n'); f.write(report)
    f.write(f'\nBest Val Accuracy: {best_val:.4f}\n')
print('Results saved.')